In [ ]:
"""
TODOS:
- BA Theorie überarbeiten
- Schauen wie jetzt die Outputs sind und regelmäßig prüfen wie sich das verändert
Inputs/Outputs sehr klar machen, alles wunderbar dokumentieren
- Fallunterscheidungen: Haben wir Dichten gegeben oder optimieren wir darüber?; Inverse ja oder nein?
- Zeile für Zeile durchgehen und übernehmen
"""

In [1]:
from safescope import safescope, Imports

with Imports(): # Necessary to have them available in functions decorated with @safescope
    from ngsolve.meshes import MakeStructured2DMesh, MakeHexMesh
    import netgen.meshing as NetgenMeshing

    #from netgen.meshing import *
    #from netgen.meshing import *
    #from netgen.csg import *
    from netgen.geom2d import unit_square, SplineGeometry
    import netgen.geom2d as geom2d

    from ngsolve import *

    #from ngsolve.krylovspace import CG,CGSolver
    from ngsolve.fem import MinimizationCF
    from ngsolve.comp import IntegrationRuleSpace
    import numpy as np
    import matplotlib.pyplot as plt

    from ngsolve.webgui import Draw

    from scipy.optimize import root, minimize
    import scipy.sparse as sp

    from ngsolve.krylovspace import GMRes

    # Helper class to create 3d mesh with thrid dimension being the time
    from OTmeshing import OTMesh

    from tqdm.notebook import trange, tqdm



In [2]:
with Imports():
    def DrawAnimate2d(gfu_in, V, mesh2d, nz):
        """
        Geht in 3D Ding die z-Achse (Zeit) hoch und nimmt jeweils 2D Slices
        TODO: Klar machen, was die jeweiligen Inputs sind.
        gfu_in:
        V: 
        mesh2d:
        nz:
        """
        gfu = GridFunction(V)
        gfu.Set(gfu_in)
        # Extract slice data from 3d mesh
        
        Vslice = H1(mesh2d, order=1)
        gfu_slice = GridFunction(Vslice)
        #gfu1d_slice = GridFunction(Vslice1d)

        ndof_slice = int(gfu.vec.size / (nz+1))

        gfut = GridFunction(gfu_slice.space,multidim=0)

        for k in range(nz+1):
            #print(k)
            gfu_slice.vec.data = gfu.vec[k*ndof_slice:(k+1)*ndof_slice] #+ gfrho1d.vec.data[k*ndof1d_slice:(k+1)*ndof1d_slice]
            
            gfut.AddMultiDimComponent(gfu_slice.vec)

        scene = Draw(gfut, mesh2d, interpolate_multidim=True, animate=True, deformation=False, height="3vh", speed=4, autoscale=False, min=min(gfu.vec), max=max(gfu.vec))
        
        return scene


In [3]:
with Imports():
    def funcderiv2(u,v, tmpr, tmpf, nrmv, alph1, alph2):
        """
        Das ist für Newton nicht auf der Kurve
        """
        fval = np.array([u**3 + alph1/alph2*u*v**2 + 2*(alph1**2+alph1*tmpr)*u - 2*alph1**2*nrmv,
                        v**3 + alph2/alph1*u**2*v + 2*(alph2**2+alph2*tmpr)*v - 2*alph2**2*tmpf])
        derivval = np.array([[3*u**2 + alph1/alph2*v**2 + 2*(alph1**2+alph1*tmpr), 2*alph1/alph2*u*v],
                            [2*alph2/alph1*u*v, 3*v**2 + alph2/alph1*u**2 + 2*(alph2**2+alph2*tmpr)]])
        return fval, derivval
    
    def funcderivInv(rhos, Jxs, Jys, alpha_inv, alpha_f, r1, etarhos, etaJxs, etaJys, rhodat, bSource = False, fs = None, etafs = None):
        """
        Das ist für Newton auf der Kurve
        """
        baralpha_inv = 2*(1/alpha_inv - alpha_inv/2)
        Jsums = Jxs + Jys
        if bSource:
            K = (1/alpha_inv-alpha_inv/2)*(rhos + 0.5*(Jxs**2 + Jys**2) + 0.5*(1/alpha_f)*fs**2)
            fval = np.array([rhodat + K - r1*etarhos + r1*rhos,
                             rhodat*Jxs + Jxs*K - r1*etaJxs + r1*Jxs,
                             rhodat*Jys + Jys*K - r1*etaJys + r1*Jys,
                             (1/alpha_f)*rhodat*fs + (fs/alpha_f)*K - r1*etafs + r1*fs])
            
            derivval = np.array([[(baralpha_inv + r1)*np.ones_like(rhos), baralpha_inv*Jsums, baralpha_inv*Jsums, (baralpha_inv/alpha_f)*fs],
                                 [baralpha_inv*Jxs, rhodat + baralpha_inv*(K+Jxs*Jsums)+r1, baralpha_inv*Jxs*Jsums, 2*(baralpha_inv/alpha_f)*Jxs*fs],
                                 [baralpha_inv*Jys, baralpha_inv*Jys*Jsums, rhodat + baralpha_inv*(K+Jys*Jsums)+r1, 2*(baralpha_inv/alpha_f)*Jys*fs],
                                 [(baralpha_inv/alpha_f)*fs, (baralpha_inv/alpha_f)*fs*Jsums, (baralpha_inv/alpha_f)*fs*Jsums, 1/alpha_f*rhodat + (baralpha_inv/alpha_f)*K + baralpha_inv*(fs/alpha_f)**2 + r1]])
        else:
            K = (1/alpha_inv-alpha_inv/2)*(rhos + 0.5*(Jxs**2 + Jys**2))
            fval = np.array([rhodat + K - r1*etarhos + r1*rhos,
                             rhodat*Jxs + Jxs*K - r1*etaJxs + r1*Jxs,
                             rhodat*Jys + Jys*K - r1*etaJys + r1*Jys])

            derivval = np.array([[(baralpha_inv + r1)*np.ones_like(rhos), baralpha_inv*Jsums, baralpha_inv*Jsums],
                                 [baralpha_inv*Jxs, rhodat + baralpha_inv*(K+Jxs*Jsums)+r1, baralpha_inv*Jxs*Jsums],
                                 [baralpha_inv*Jys, baralpha_inv*Jys*Jsums, rhodat + baralpha_inv*(K+Jys*Jsums)+r1]])
            
        return fval, derivval


### Parameters to select type of problem

In [4]:
#Parameter setzen und erst einmal definieren
bWind = True
bSource = True
bInitialFinial = True
bInverse = True

alpha_f = 1
alpha_inv = 1
alpha_wind = 1

vtk_output = False

### Create Meshes and Spaces

In [5]:
# Create 2d and 1d meshes
otmesh = OTMesh(h=0.05, nz = 40, holes = False)

mesh = otmesh.mesh3d
mesh2d = otmesh.mesh2d
nz = otmesh.nz

order=1
orderm=1
orderI=1

# H1 ist stückweise linear, L2 ist stückweise konstant. TODO Dazu noch anschauen was finite Elemente sind/ wie die funktionieren 
V = H1(mesh, order=orderm) ## FIXME: Dirichlet conditions? -> do not change things
W = L2(mesh, order=0) #IntegrationRuleSpace(mesh, order=orderm) ## FIXME: Dirichlet conditions? -> do not change things
ndof3d = W.ndof #TODO ndof nochmal klar machen
Draw(mesh)
#W = IntegrationRuleSpace(mesh, order=orderm) ## FIXME: Dirichlet conditions? -> do not change things
#irs_dx = dx(intrules=W.GetIntegrationRules())

Initial number of points =  511


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

### Set-up inverse problem: Set sensor locations 

In [6]:
np_xy = 30
sensor_points_xy = [ (0.7,0.25+0.5*i/np_xy) for i in range(np_xy) ] # Must have length nz

sensor_points = []
for i in range(2*nz):
    for p in sensor_points_xy:
        sensor_points.append([p[0], p[1], i/(2*nz)])
        sensor_points.append([p[0]-otmesh.h/2, p[1], i/(2*nz)])
        sensor_points.append([p[0]+otmesh.h/2, p[1], i/(2*nz)])

sensor_marker = BitArray(mesh.ne)  # mesh.ne = number of elements
sensor_marker.Clear()

for pt in sensor_points:
    el_nr = mesh(pt[0],pt[1],pt[2]) # mesh2d(pt[0],pt[1]) = 2d point
    sensor_marker.Set(el_nr.nr) # with i the number of the element
    
# Set up inverse problem data
gfInvDat = GridFunction(W)
gfInvDat.Set(IfPos(z-.1, IfPos(.9-z,z*(1-z),0),0), definedonelements=sensor_marker)
gfInvDat.vec.data /= Integrate(gfInvDat, mesh)    
gfInvDat.vec.data *= .5

DrawAnimate2d(gfInvDat, V, mesh2d, nz)

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

### Solve Stokes problem to create "wind" fields

Das ist für das Windfeld, kann eigentlich erst einmal ignoriert werden

In [7]:
@safescope
def SolveStokes(mesh2d, alpha_wind = 1, mu_val=1):
    V = VectorH1(mesh2d, order=1, dirichlet="curve|inflow")
    Q = H1(mesh2d, order=1)
    X = V*Q
    
    (u,p),(v,q) = X.TnT()

    stokes = mu_val*InnerProduct(Grad(u), Grad(v))*dx + div(u)*q*dx + div(v)*p*dx
    a = BilinearForm(stokes).Assemble()

    gf = GridFunction(X)
    gfu, gfp = gf.components

    #uin = CF ( (1.5*4*y*(0.41-y)/(0.41*0.41), 0, 0) )
    #uin = CF ( ((1-0*.5*y), 0, 0) )
    uin = alpha_wind*CF ( ((1-0*.5*y), 0) )
    gfu.Set(uin, definedon=mesh2d.Boundaries("inflow"))

    res = -a.mat * gf.vec
    inv = a.mat.Inverse(freedofs=X.FreeDofs()) #, inverse="pardiso")
    gf.vec.data += inv * res

    #Draw(gfu)

    return gfu

gfWind = SolveStokes(mesh2d, alpha_wind=alpha_wind, mu_val=1)

Draw(gfWind,height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

### Create Spaces and Data

In [8]:
tol0=1e-10
iterFlag=10
sol = "direct"
eps0=0

n = specialcf.normal(mesh.dim)

def CreateVariables(VV, WW):#, WW):
    # "testfunction"
    ph = GridFunction(VV)
    # primal variables
    dens = GridFunction(WW)  
    rho0 = GridFunction(WW)  
    rho1 = GridFunction(WW)  
    flux1 = GridFunction(WW)
    flux2 = GridFunction(WW)
    # dual variables
    denss = GridFunction(WW)  
    flux1s = GridFunction(WW)
    flux2s = GridFunction(WW)
    
    # derivative of testfunction as GridFunction not CF
    dph = GridFunction(WW*WW*WW) # Gradient

    return ph, dens, rho0, rho1, flux1, flux2, denss, flux1s, flux2s, dph 
    
# Set-up for bulk problem

# rhoEx = exp(-1/2/sigma**2*tt1) + exp(-1/2/sigma**2*tt2) + eps0 #exp(-1/2/sigma**2*tt1) + 1/2*exp(-1/2/sigma**2*tt2) + eps0 #exp(-1/2/(sigma+0.1*z)**2*tt)+eps0 # Exact density
# massinit = Integrate(rhoEx, mesh, definedon=mesh.Boundaries("bottom"))
# rhoEx *= 1/2*1/massinit
# # rhoEx = exp(-1/2/(sigma)**2*tt)+eps0 # Exact density
# mEx = CF((0.5*rhoEx, 0.5*rhoEx)) # Exact flux
# rho0 = rhoEx #exp(-1/2/(sigmab)**2*ttb)+eps0#rhoEx
# rho1 = rhoEx #exp(-1/2/(sigmab)**2*ttb)+eps0 # 0*rhoEx #rhoEx #
phiEx = 0.5*(x+y)-0.25*y

phih, rho, rho0, rho1, Jx, Jy, rhos, Jxs, Jys, dphih = CreateVariables(V, W)


lam_avg = 0 # Lagrange multiplier for RHS of Poisson to integrate to zero

#TODO ???????
source = GridFunction(W)
sources = GridFunction(W)

vWindX = GridFunction(W)
vWindY = GridFunction(W)
# Set up "wind" vectorfield
if bWind:
    vWindX.Set(gfWind.components[0])
    vWindY.Set(gfWind.components[1])
else:
    vWindX.Set(0*x)
    vWindY.Set(0*x)
    
# Set up initial- and final mass distributions
sigma = .1
#sigma2 = .05

#tt3 =(x-(.3*(1-z)+.6*z))**2+(y-(.3*(1-z)+.6*z))**2
#tt3 =(x-(.5))**2+(y-(.5))**2
#tt3 =(x-(.3))**2+(y-(.2*(1-z)+.8*z))**2
tt3 =(x-(.35*(1-z)+.7*z))**2+(y-(.35*(1-z)+.7*z))**2

# tt1 = (x-.2-.6*z)**2+(y-.2-.3*z)**2
# tt2a = (x-.2)**2+(y-.8-.5*(1-z))**2
# tt2b = (x-.4)**2+(y-.8-.5*(1-z))**2
# tt2c = (x-.6)**2+(y-.8-.5*(1-z))**2
# rhoEx1 = exp(-1/2/sigma**2*tt1)
# rhoEx2 = exp(-1/2/sigma2**2*tt2a) + exp(-1/2/sigma2**2*tt2b) + exp(-1/2/sigma2**2*tt2c)
rhoEx = 0*exp(-(1/2/sigma**2)*tt3) + 0*1 # FIXME

# gfEx = GridFunction(V)
# # gfEx.Set(rhoEx1 + rhoEx2)
# #gfEx.Interpolate(rhoEx) # 
# gfEx.Set(rhoEx)

# gfEx.vec.data = gfEx.vec
# rhoEx = CF((gfEx))

if bInitialFinial:
    rho0cf = rhoEx
    rho1cf = rhoEx

    # masstop = Integrate(rho1cf, mesh, definedon=mesh.Boundaries("top"))
    # massbottom = Integrate(rho0cf, mesh, definedon=mesh.Boundaries("bottom"))

    # # Normalize to make probability densities
    # rho0cf *= 1/massbottom
    # rho1cf *= 1/masstop
else:
    rho0 = GridFunction(W)
    rho1 = GridFunction(W)

mEx = CF((.5*rhoEx, .5*rhoEx))

### Subroutines for ALG2-Solver

In [9]:
@safescope
def assembleBilinearForm(V, r1, bWind, bSource, vWindX, vWindY, tol=1e-4):
    """
    Brauchen wir für das phi 
    """
    # Tried dividing by r1 as well, did not change 
    (phi, psi) = V.TnT()

    ## Bilinear form
    aprod = BilinearForm(V, condense=True)
    aprod += r1*grad(phi)*grad(psi)*dx #irs_dx ## FIXME the scaling factor in front might need to be 1/hmesh*
    
    if bWind:
        aprod += r1*(CF((vWindX, vWindY))*CF((grad(phi)[0],grad(phi)[1])))*grad(psi)[2]*dx
        aprod += r1*(CF((vWindX, vWindY))*CF((grad(psi)[0],grad(psi)[1])))*grad(phi)[2]*dx
        aprod += r1*(CF((vWindX, vWindY))*CF((grad(phi)[0],grad(phi)[1])))*(CF((vWindX, vWindY))*CF((grad(psi)[0],grad(psi)[1])))*dx
        
    if bSource:
        aprod += r1*phi*psi*dx #irs_dx # Uniqueness/stability term
    else:
        aprod += tol*phi*psi*dx #irs_dx # Uniqueness/stability term
  
    # precon = Preconditioner( aprod, "bddc", inverse="pardiso")
    aprod.Assemble()
    invaprod = aprod.mat.Inverse(V.FreeDofs(True), inverse="pardiso") ## FIXME: Use non-symmetric solver!!!
    
    return invaprod

@safescope
def SolvePhi(V, r1, rho0, rho1, rho, Jx, Jy, rhos, Jxs, Jys, source, sources, invaprod, bWind, bSource, bInitialFinial, vWindX = 0, vWindY = 0):
    
    psi = V.TestFunction()                                                                                                                                                  
    fbulk = LinearForm(V)
    # bulk rhs
    #muhvec = CF((Jx + vWindX*rho, Jy + vWindY*rho, rho))
    muhvec = CF((Jx, Jy, rho))
    qhvec = CF((Jxs, Jys, rhos))

    # bulk rhs    
    fbulk += (r1*qhvec - 1*muhvec)*grad(psi)*dx # irs_dx # gradient part; Jx and Jy ## FIXME the scaling factor infront might need to be 1/hmesh*
    
    if bWind:
        fbulk += (r1*CF((vWindX, vWindY))*CF((grad(psi)[0],grad(psi)[1])))*(r1*rhos-rho)*dx
    
    if bSource:
        fbulk += (r1*sources - source)*psi*dx # irs_dx # source term

    if bInitialFinial:
        mesh = V.mesh
        # Hier oberen und unteren Rand des Würfels definieren
        fbulk += (-1)*rho0*psi.Trace()*ds(definedon=mesh.Boundaries("bottom")) # implements initial condition
        fbulk += 1*rho1*psi.Trace()*ds(definedon=mesh.Boundaries("top")) # implements final condition
    
    fbulk.Assemble()
    
    newvec = GridFunction(V)
    gfuprod = GridFunction(V)
    
    newvec.vec.data = fbulk.vec
    gfuprod.vec.data = invaprod * newvec.vec
    
    return gfuprod.vec


@safescope
def UpdatePhi(r1, rho, dphih, phiL2, Jx, Jy, source, rhos, Jxs, Jys, sources, bWind, bSource, bInverse, sensor_marker, alpha_f, alpha_inv, vWindX, vWindY, gfInvDat):

    if bInverse:
        indNot = sensor_marker
    else:
        indNot = BitArray(rho.vec.size)
        indNot.Clear()

    etarhos = dphih.components[2].vec.FV().NumPy() + 1/r1*rho.vec.FV().NumPy()
    
    if bWind:
        etarhos += dphih.components[0].vec.FV().NumPy()*vWindX.vec.FV().NumPy() + dphih.components[1].vec.FV().NumPy()*vWindY.vec.FV().NumPy()
        
    etaJxs = dphih.components[0].vec.FV().NumPy() + 1/r1*Jx.vec.FV().NumPy()
    etaJys = dphih.components[1].vec.FV().NumPy() + 1/r1*Jy.vec.FV().NumPy()

    if bSource:
        etafs = phiL2.vec.FV().NumPy() + 1/r1*source.vec.FV().NumPy()

        ## Check admissibility bulk.
        PosIndexG = etarhos + .5*(etaJxs**2 + etaJys**2 + etafs**2/alpha_f) > 0
        ## Update dual variables curve
        if np.any(PosIndexG)==True:
            ag = etarhos[PosIndexG].copy()
            bgx = etaJxs[PosIndexG].copy()
            bgy = etaJys[PosIndexG].copy()
            cg = etafs[PosIndexG].copy()
            nrm_b1d = (bgx**2+bgy**2)**0.5
            # Solve the system using Newton's method
            # b0x = bgx.copy()
            # b0y = bgy.copy()
            # c0 = cg.copy()
            mu01d = nrm_b1d.copy()
            nu01d = cg.copy()
            err1d = 1
            for i in range(np.shape(ag)[0]):
                if nrm_b1d[i]==0:
                    print("zero encountered")
                    nrm_b1d[i] = 1
                    mu01d[i] = 0
                    funcderivtest = lambda c: np.array([c**3 + 2*(alpha_f**2+alpha_f*ag[i])*c - 2*alpha_f**2*cg[i], 3*c**2 + 2*(alpha_f**2+alpha_f*ag[i])])
                    solroottest = root(funcderivtest, nu01d[i], jac=True, method='hybr')
                    nu01d[i] = solroottest.x[0]
                    print(solroottest.message)
                # elif fcurves.vec[i]==0:
                #     # print("no dual flux")
                #     nu01d[i] = 0
                #     funcderivbulk = lambda c: np.array([c**3 + 2*(ag[i]+1)*c - 2*nrm_b1d[i], 3*c**2 + 2*(ag[i]+1)])
                #     solrootbulk = root(funcderivbulk, mu01d[i], jac=True,  method='hybr')
                #     mu01d[i] = solrootbulk.x[0]
                else:
                    funcderivparam2 = lambda c: funcderiv2(c[0], c[1], ag[i], cg[i], nrm_b1d[i], 1, alpha_f) 
                    zold = np.array([mu01d[i], nu01d[i]])
                    solrootcurve = root(funcderivparam2, zold, jac=True, method='hybr')
                    #print(solrootcurve.message)
                    znew = solrootcurve.x
                    mu01d[i] = znew[0]
                    nu01d[i] = znew[1]
            etarhos[PosIndexG] = -0.5*(mu01d**2 + nu01d**2/alpha_f)
            etaJxs[PosIndexG] = mu01d*bgx/nrm_b1d
            etaJys[PosIndexG] = mu01d*bgy/nrm_b1d
            etafs[PosIndexG] = nu01d ## FIXME    
        

        #sources.vec.data[~indNot] = etafs[~indNot]
        sources.vec.data = etafs
        
    else: # Case without source
        PosIndexB = etarhos + .5*(etaJxs**2 + etaJys**2) > 0
        iter0 = 0
        
        ## Update dual variables bulk
        if np.any(PosIndexB)==True:
            # print("there are non admissible in bulk")
            a = etarhos[PosIndexB].copy()
            b1 = etaJxs[PosIndexB].copy()
            b2 = etaJys[PosIndexB].copy()
            nrm_b = (b1**2+b2**2)**0.5
            # FIND (unique) largest real root of cubic poly
            # x^3 + 2(alpha+1) x - 2 |beta| = 0
            # Use Newton's method.
            mu0 = nrm_b.copy()
            err = 1
            # for i in range(np.shape(a)[0]):
                # if nrm_b[i]==0:
                #     print("zero encountered in bulk")
                #     nrm_b[i] = 1
                #     mu0[i] = 0
                # else:
                #     funcderivbulk = lambda c: np.array([c**3 + 2*(a[i]+1)*c - 2*nrm_b[i], 3*c**2 + 2*(a[i]+1)])
                #     solrootbulk = root(funcderivbulk, mu0[i], jac=True,  method='hybr')
                #     mu0[i] = solrootbulk.x[0]
            while err > 1e-6 and iter0<20:
                iter0 += 1
                mu1 = mu0 - (mu0**3 + 2*(a+1)*mu0 - 2*nrm_b)/(3*mu0**2+2*(a+1)) # x = x - f(x)/f'(x)
                err = abs(mu1-mu0).max()
                mu0 = mu1
                
            etarhos[PosIndexB] = -0.5*mu0**2
            etaJxs[PosIndexB] = mu0*b1/nrm_b
            etaJys[PosIndexB] = mu0*b2/nrm_b
   
    #rhos.vec.data[~indNot] = etarhos[~indNot]
    #Jxs.vec.data[~indNot] = etaJxs[~indNot]  
    #Jys.vec.data[~indNot] = etaJys[~indNot]
    
    rhos.vec.data = etarhos
    Jxs.vec.data = etaJxs
    Jys.vec.data = etaJys

    
    if bInverse:
        for i in range(len(indNot)):
            if indNot[i]:
                if bSource:
                    funcderivInv2 = lambda c: funcderivInv(c[0], c[1], c[2], alpha_inv, alpha_f, r1, etarhos[i], etaJxs[i], etaJys[i],  gfInvDat.vec.FV().NumPy()[i], bSource=True, fs = c[3], etafs = etafs[i])
                    
                    zold = np.array([rhos.vec.FV().NumPy()[i], Jxs.vec.FV().NumPy()[i], Jys.vec.FV().NumPy()[i], sources.vec.FV().NumPy()[i]])
                else:
                    funcderivInv2 = lambda c: funcderivInv(c[0], c[1], c[2], alpha_inv, alpha_f, r1, etarhos[i], etaJxs[i], etaJys[i], gfInvDat.vec.FV().NumPy()[i]) 

                    zold = np.array([rhos.vec.FV().NumPy()[i], Jxs.vec.FV().NumPy()[i], Jys.vec.FV().NumPy()[i]])
                
                solrootcurve = root(funcderivInv2, zold, jac=True, method='hybr')        
                
                rhos.vec.data[i] = solrootcurve.x[0]
                Jxs.vec.data[i] = solrootcurve.x[1]
                Jys.vec.data[i] = solrootcurve.x[2]
                if bSource:
                    sources.vec.data[i] = solrootcurve.x[3]
            
    return 0

def interpolatedPhi(phih, dphih):
    dphih.components[0].Set(grad(phih)[0])
    dphih.components[1].Set(grad(phih)[1])
    dphih.components[2].Set(grad(phih)[2])

    return 0


### Set initial data

In [10]:
# initial data
rho.Interpolate(rhoEx) # .Set produces neg. values -> why?
Jx.Set(0*mEx[0]) 
Jy.Set(0*mEx[1]) 
rhos.Set(0*phiEx.Diff(z))
Jxs.Set(0*phiEx.Diff(x))
Jys.Set(0*phiEx.Diff(y))
source.Set(0*x)
if not bInitialFinial:
    rho0.Set(0*x)
    rho1.Set(0*x)

In [11]:
def Alg2_solve(V, W, r1, rho, rho0, rho1, Jx, Jy, rhos, Jxs, Jys, source, sources, ndof3d, invaprod, bWind, bSource, bInitialFinial, vWindX, vWindY, scaler1=1, iterMax=100, tol=1e-6):

    iter0 = 0
    duality_gaps = []
    phiL2 = GridFunction(W)
    lam_avg = 0

    err0 = 1 #scaler1*r1*max(np.abs(dphih.components[2].vec.FV().NumPy() - rhos.vec.FV().NumPy()).max(), 
            #    np.abs(dphih.components[0].vec.FV().NumPy() - Jxs.vec.FV().NumPy()).max(), 
            #    np.abs(dphih.components[1].vec.FV().NumPy() - Jys.vec.FV().NumPy()).max())
    
   
    # minerr = err0
    tau = 1

    with TaskManager():
        while iter0 < iterMax+1:

            ## STEP A: Solve for new (phi,phi1d)
            if bInitialFinial:
                phih.vec.data = SolvePhi(V, r1, rho0cf, rho1cf, rho, Jx, Jy, rhos, Jxs, Jys, source, sources, invaprod, bWind, bSource, bInitialFinial, vWindX, vWindY)
            else:
                phih.vec.data = SolvePhi(V, r1, rho0, rho1, rho, Jx, Jy, rhos, Jxs, Jys, source, sources, invaprod, bWind, bSource, bInitialFinial, vWindX, vWindY)                
            
            phiL2.Set(phih)            
            
            lam_avg = lam_avg + tau*r1*(Integrate(rho1, mesh, definedon=mesh.Boundaries("top")) - Integrate(rho0, mesh, definedon=mesh.Boundaries("bottom")))
            if bSource:
                lam_avg += tau*r1*bSource*Integrate(source, mesh)

            ## STEP B: solve pointwise nonlinear eqn (USE NEWTON)
            interpolatedPhi(phih, dphih)            
            UpdatePhi(r1, rho, dphih, phiL2, Jx, Jy, source, rhos, Jxs, Jys, sources, bWind, bSource, bInverse, sensor_marker, alpha_f, alpha_inv, vWindX, vWindY, gfInvDat)
            #sources.vec.FV().NumPy()[:] = (source.vec.FV().NumPy()[:] + r1*phiL2.vec.FV().NumPy()) / (r1 + (1./alpha_f))
            
            ## STEPC: 
            # update primal variables on bulk
            #rho.vec.data += scaler1*r1*(dphih.components[2].vec - rhos.vec + vWindX*dphih.components[0].vec + vWindY*dphih.components[1].vec)#(dphih.components[2].vec.FV().NumPy() - rhos.vec.FV().NumPy())
            #rho.vec.FV().NumPy()[:] += scaler1*r1*(dphih.components[2].vec.FV().NumPy() - rhos.vec.FV().NumPy() + vWindX.vec.FV().NumPy()*dphih.components[0].vec.FV().NumPy() + vWindY.vec.FV().NumPy()*dphih.components[1].vec.FV().NumPy())#(dphih.components[2].vec.FV().NumPy() - rhos.vec.FV().NumPy())
            rho.vec.FV().NumPy()[:] += tau*r1*(dphih.components[2].vec.FV().NumPy() + vWindX.vec.FV().NumPy()*dphih.components[0].vec.FV().NumPy() + vWindY.vec.FV().NumPy()*dphih.components[1].vec.FV().NumPy() - rhos.vec.FV().NumPy())
              
            Jx.vec.data += tau*r1*(dphih.components[0].vec - Jxs.vec)#(dphih.components[0].vec.FV().NumPy() - Jxs.vec.FV().NumPy())
            Jy.vec.data += tau*r1*(dphih.components[1].vec - Jys.vec)#(dphih.components[1].vec.FV().NumPy() - Jys.vec.FV().NumPy())
            

            if not bInitialFinial: # update initial- and final data
                rho0.vec.FV().NumPy()[:] += tau*r1*(-phiL2.vec.FV().NumPy() + lam_avg)
                rho1.vec.FV().NumPy()[:] += tau*r1*(phiL2.vec.FV().NumPy() - lam_avg)
            
            if bSource:
                source.vec.FV().NumPy()[:] += tau*r1*(phiL2.vec.FV().NumPy() - sources.vec.FV().NumPy() - lam_avg) 
                                
            
            if iter0%iterFlag==0:
                err0 = max(np.abs(dphih.components[2].vec.FV().NumPy() + dphih.components[0].vec.FV().NumPy()*vWindX.vec.FV().NumPy() + dphih.components[1].vec.FV().NumPy()*vWindY.vec.FV().NumPy() - rhos.vec.FV().NumPy()).max(),np.abs(dphih.components[0].vec.FV().NumPy() - Jxs.vec.FV().NumPy()).max(),np.abs(dphih.components[1].vec.FV().NumPy() - Jys.vec.FV().NumPy()).max())
                if bSource:
                    err0 = max(err0, np.abs(phiL2.vec.FV().NumPy() - sources.vec.FV().NumPy()).max())
                
                rhs_gap = (Integrate(rho1, mesh, definedon=mesh.Boundaries("top")) - Integrate(rho0, mesh, definedon=mesh.Boundaries("bottom")) + Integrate(source, mesh))

                print(f"iter =  {iter0} / {maxiter}\n rhs integral =  {rhs_gap} \n lam_avg = {lam_avg} \n duality gap = {err0}\n gap in f^* = {np.abs(phiL2.vec.FV().NumPy() - sources.vec.FV().NumPy()).max()} \n source integral = {Integrate(source, mesh)} \n Gap in rho {np.abs(dphih.components[2].vec.FV().NumPy() - rhos.vec.FV().NumPy() + vWindX.vec.FV().NumPy()*dphih.components[0].vec.FV().NumPy() + vWindY.vec.FV().NumPy()*dphih.components[1].vec.FV().NumPy()).max()}")
                #print("inverse error",(rho.vec.FV().NumPy()[sensor_marker] - gfInvDat.vec.FV().NumPy()[sensor_marker]).max(), "Gap in rho", np.abs(dphih.components[2].vec.FV().NumPy() - rhos.vec.FV().NumPy() + vWindX.vec.FV().NumPy()*dphih.components[0].vec.FV().NumPy() + vWindY.vec.FV().NumPy()*dphih.components[1].vec.FV().NumPy()).max(), end="\r")
                
                
                duality_gaps.append(err0)
                #print(err0)
                # if err0<minerr:
                #     minerr = err0
                # elif err1>1.5*minerr:
                #     print("changes bulk")
                #     rho.vec.data += -1.5*scaler1*r1*(dphih.components[2].vec.FV().NumPy() - rhos.vec.FV().NumPy())
                #     Jx.vec.data += -1.5*scaler1*r1*(dphih.components[0].vec.FV().NumPy() - Jxs.vec.FV().NumPy())
                #     Jy.vec.data += -1.5*scaler1*r1*(dphih.components[1].vec.FV().NumPy() - Jys.vec.FV().NumPy())
                # if err01d<minerr1d:
                #     minerr1d = err01d
                # elif err11d>1.5*minerr1d:
                #     print("changes curve")
                #     rho1d.vec.data += -1.5*scaler2*r2*(dphih1dproj2.vec.FV().NumPy() - rho1ds.vec.FV().NumPy())
                #     J1d.vec.data += -1.5*scaler2*r2*(dphih1dproj0.vec.FV().NumPy() + dphih1dproj1.vec.FV().NumPy() - J1ds.vec.FV().NumPy()) #((dhih1dproj[:ndof1d] + dhih1dproj[ndof1d:2*ndof1d]) - J1ds.vec.FV().NumPy()) # FIXME: normalized projection of the gradient :)
                #     fcurve.vec.data += -1.5*scaler2*r2*(projCurve(phih1d, W1d).vec.FV().NumPy() - projCurve(phih, W1d).vec.FV().NumPy() - fcurves.vec.FV().NumPy())

                if err0 < tol: # FIXME 
                    print("converged")
                    return rho, rho0, rho1, phih, Jx, Jy, Jxs, Jys, source, sources, iter0, err0, duality_gaps



            iter0 += 1
        print("max number of iterations reached.") 

    return rho, rho0, rho1, phih, Jx, Jy, Jxs, Jys, source, sources, iter0, err0, duality_gaps

r1 = 1 #/hmesh**3 #(alpha1*alpha2*hmesh**3) #1/hmesh**3 #100000 #
#r2 = 1 #*(alpha1*alpha2)
sr1 = 1 #/r1 # to get rid of the scaling problem coming from the update step -> does not converge
#sr2 = 1

toldiff = 1e-4
tolcvg = 1e-5
maxiter = 500 # 50000

invaprod = assembleBilinearForm(V, r1, bWind, bSource, vWindX, vWindY, tol=toldiff)

rho, rho0, rho1, phih, Jx, Jy, Jxs, Jys, source, Thesources, iter0, err0, duality_gaps = Alg2_solve(V, W, r1, rho, rho0, rho1, Jx, Jy, source, sources, rhos, Jxs, Jys, ndof3d, invaprod, bWind, bSource, bInitialFinial, vWindX, vWindY, scaler1=sr1, iterMax=maxiter, tol=tolcvg)

#Draw(rho)
#Draw(source)
#Draw(phih)

max number of iterations reached.=  7.653285248275866e-07 lam_avg =  -0.0001102704730121278  duality gap =  1.3380044802356572  gap in f^* =  1.3380044802356572 source integral =  7.653285269280818e-07 Gap in rho 0.2897130725238234480625


In [12]:
gfrho = GridFunction(V)
gfrho.Set(rho)

# gfFlux = GridFunction(W)
# gfFlux.components[0].Set(Jx)
# gfFlux.components[1].Set(Jy)

print(Integrate(gfrho, mesh, definedon=mesh.Boundaries("top")))
print(Integrate(gfrho, mesh, definedon=mesh.Boundaries("bottom")))

# Extract slice data from 3d mesh
mesh2d = otmesh.mesh2d

Vslice = H1(mesh2d, order=1)
VsliceFlux = VectorH1(mesh2d, order=1)
#Vslice1d = H1(mesh2d, order=1)


print("ndof Vslice = ", Vslice.ndof)
#print("ndof Vslice1d = ", Vslice1d.ndof)

gfu_slice = GridFunction(Vslice)
gfuFlux_slice = GridFunction(VsliceFlux)
#gfu1d_slice = GridFunction(Vslice1d)

ndof_slice = int(gfrho.vec.size / (nz+1))
#ndofFlux_slice = int(Jx.vec.size / (nz))
print("ndof_slice = ", ndof_slice)
#ndof1d_slice = int(gfrho1d.vec.size / (nz+1))
#print("ndof1d_slice = ", ndof1d_slice)

gfut = GridFunction(gfu_slice.space,multidim=0)
gfutFlux = GridFunction(gfuFlux_slice.space,multidim=0)
#gfut1d = GridFunction(gfu1d_slice.space,multidim=0)

Jx_proj = GridFunction(H1(mesh, order=1))
Jx_proj.Set(Jx)
Jy_proj = GridFunction(H1(mesh, order=1))
Jy_proj.Set(Jy)
source_proj = GridFunction(H1(mesh, order=1))
source_proj.Set(source)

if vtk_output:
    vtk = VTKOutput(ma=mesh2d,coefs=[gfu_slice],names=["bulk"],filename="vtk/bulk",subdivision=4)
#vtk.Do()
#gfut.AddMultiDimComponent(gfEx.vec)
for k in range(nz+1):
    #print(k)
    #gfu_slice.vec.data = gfrho.vec[k*ndof_slice:(k+1)*ndof_slice] #+ gfrho1d.vec.data[k*ndof1d_slice:(k+1)*ndof1d_slice]
    
    
    gfu_slice.vec.data = source_proj.vec[k*ndof_slice:(k+1)*ndof_slice] #+ gfrho1d.vec.data[k*ndof1d_slice:(k+1)*ndof1d_slice]
    #print("Mass at time ", k, " = ", Integrate(gfu_slice, mesh2d))
    gfut.AddMultiDimComponent(gfu_slice.vec)
    # PlotSolution(gfu_slice, mesh2d, title="rho", fig=fig)
# for k in range(nz):
#     gfu_slice.vec.data = gfrho.vec.data[k*ndof_slice:(k+1)*ndof_slice]
#     gfu_slice.vec.data += gfrho.vec.data[(k+1)*ndof_slice:(k+2)*ndof_slice] 
#     gfu_slice.vec.data *= 0.5
    
    # Below should yield velocity but does not, probably due to division by (almost) zero
    # gfuFlux_slice.components[0].vec.data.FV().NumPy()[:] = Jx_proj.vec[k*ndof_slice:(k+1)*ndof_slice].FV().NumPy()[:] / np.maximum(gfu_slice.vec.FV().NumPy(),0.0001)
    # gfuFlux_slice.components[1].vec.data.FV().NumPy()[:] = Jy_proj.vec[k*ndof_slice:(k+1)*ndof_slice].FV().NumPy()[:] / np.maximum(gfu_slice.vec.FV().NumPy(),0.0001)

    gfuFlux_slice.components[0].vec.data = Jx_proj.vec[k*ndof_slice:(k+1)*ndof_slice] # .FV().NumPy() #/ gfu_slice_proj.vec.FV().NumPy()
    gfuFlux_slice.components[1].vec.data = Jy_proj.vec[k*ndof_slice:(k+1)*ndof_slice] # .FV().NumPy() #/ gfu_slice_proj.vec.FV().NumPy()
    #gfuFlux_slice.components[0].vec.FV().NumPy()[:] = vWindX*gfu_slice_proj.vec.FV().NumPy()
    #gfuFlux_slice.components[1].vec.FV().NumPy()[:] = vWindY*gfu_slice_proj.vec.FV().NumPy()
    
    gfutFlux.AddMultiDimComponent(gfuFlux_slice.vec)
    
    #gfu1d_slice.vec.data = gfrho1d.vec.data[k*ndof1d_slice:(k+1)*ndof1d_slice]
    #gfut1d.AddMultiDimComponent(gfu1d_slice.vec)
    if vtk_output:
        vtk.Do()

# Careful, there may be a bug in autoscale, especially when working remote!!!!!
# -> Here: use fixed min/max values, autoscale=False, min=0, max=.25
#scene = Draw(gfut, mesh2d, interpolate_multidim=True, animate=True, deformation=False, height="3vh", autoscale=False, min=min(rho.vec), max=max(rho.vec))
scene = Draw(gfut, mesh2d, interpolate_multidim=True, animate=True, deformation=False, height="3vh")
#scene = Draw(gfutFlux, mesh2d, interpolate_multidim=True, animate=True, deformation=False, height="3vh")
#scene = Draw(otmesh.mesh3d)
#clear_output()
#scene.Draw(height="3vh")
#Draw(gfut1d, mesh2d, interpolate_multidim=True, animate=True, deformation=False)

0.4430521033505858
0.4496662358954692
ndof Vslice =  511
ndof_slice =  511


WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…